# Z5007 — Week 04: Simple Sorting

A self-study notebook on bubble sort, selection sort, insertion sort,
counting sort, and radix sort — how they work, their complexity, and when
"simple" sorts genuinely beat the fancier ones.


## Learning Objectives

By the end of this notebook you will be able to:

- Implement bubble sort, selection sort, and insertion sort from scratch and trace them by hand.
- Derive and explain the best/worst/average-case time complexity of each simple sort.
- Explain stability (does a sort preserve the relative order of equal elements?) and identify which of the simple sorts are stable and which are not.
- Implement counting sort and radix sort and explain why they escape the O(n log n) comparison-sort lower bound.
- Empirically measure comparison and swap counts for the simple sorts on real inputs, rather than relying on memorized formulas.
- Judge, for a given dataset (size, existing order, key range), which simple sort — if any — is actually the right choice.


## How to use this notebook

Run the cells top to bottom. Markdown cells explain a concept; the code cell
right after it is a fully worked, heavily commented example. Cells marked
`# TODO` in **Exercises** are for you to complete — replace
`raise NotImplementedError` with your own code. Each exercise is followed by
an assert-based **Self-Check** cell: it raises `AssertionError` if your code
is wrong, and prints a success message if it is right. Solutions are
collected at the end — attempt the exercises first.


## 1. Bubble sort

Bubble sort repeatedly scans the array, swapping adjacent elements that are
out of order. After each full pass, the largest remaining unsorted element
has "bubbled up" to its correct position at the end, so each pass can safely
shrink by one. A `swapped` flag lets the sort stop early if a pass makes no
swaps at all — meaning the array is already sorted, which turns the best
case (already-sorted input) into O(n) instead of O(n^2).

**Complexity:** O(n^2) comparisons and swaps in the worst and average case,
O(n) in the best case (already sorted, thanks to the early-exit flag).
**In-place:** yes. **Stable:** yes — equal elements are only ever swapped
when strictly out of order, so two equal elements never cross.


In [ ]:
def bubble_sort(arr):
    arr = arr[:]           # copy: never mutate the caller's list silently
    n = len(arr)
    for i in range(n - 1):                    # after pass i, the last i elements are final
        swapped = False
        for j in range(n - 1 - i):            # shrink the scanned range each pass
            if arr[j] > arr[j + 1]:           # strictly greater: equal elements never swap (stability)
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:                       # no swaps this pass => already sorted, stop early
            break
    return arr


assert bubble_sort([5, 2, 4, 1, 3]) == [1, 2, 3, 4, 5]
assert bubble_sort([]) == []
assert bubble_sort([1]) == [1]
assert bubble_sort([1, 2, 3, 4, 5]) == [1, 2, 3, 4, 5]   # already sorted: early exit path
print("Bubble sort checks passed")


## 2. Selection sort

Selection sort repeatedly finds the minimum of the *unsorted* remainder and
swaps it into place at the front. Unlike bubble sort, it always does
exactly n-1 passes and roughly n^2/2 comparisons — there is no early exit,
because finding the minimum requires scanning the whole remainder regardless
of how sorted it already is.

**Complexity:** O(n^2) comparisons in every case (best, worst, average are
all the same) — but only O(n) swaps, one per pass, which matters when swaps
are expensive (e.g. large records). **In-place:** yes. **Stable:** no, in
general — swapping the found minimum into position `i` can jump it past an
equal element that was originally closer to the front.


In [ ]:
def selection_sort(arr):
    arr = arr[:]
    n = len(arr)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):             # scan the ENTIRE unsorted remainder every time
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]   # one swap per pass, even if i == min_idx
    return arr


assert selection_sort([5, 2, 4, 1, 3]) == [1, 2, 3, 4, 5]
assert selection_sort([]) == []
assert selection_sort([2, 2, 1]) == [1, 2, 2]

# Demonstrate instability directly: tag equal keys so we can see if their
# relative order survives.
def selection_sort_tagged(pairs):
    pairs = pairs[:]
    n = len(pairs)
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if pairs[j][0] < pairs[min_idx][0]:
                min_idx = j
        pairs[i], pairs[min_idx] = pairs[min_idx], pairs[i]
    return pairs

tagged = [(3, "a"), (3, "b"), (1, "c")]
result = selection_sort_tagged(tagged)
# The minimum (1, "c") swaps into position 0 with (3, "a"), which JUMPS
# (3, "a") past (3, "b") -- so "a" no longer precedes "b": NOT stable.
assert result == [(1, "c"), (3, "b"), (3, "a")]
print("Selection sort checks passed; instability demonstrated on tagged input")


## 3. Insertion sort

Insertion sort builds up a sorted prefix one element at a time: it takes the
next unsorted element (the "key") and shifts elements in the sorted prefix
rightward until it finds the key's correct spot. This is exactly how most
people sort a hand of playing cards.

**Complexity:** O(n^2) worst case (reverse-sorted input: every insertion
shifts everything before it), but O(n) best case (already-sorted input:
the inner `while` never shifts anything) — and, importantly, its cost scales
with how far each element is from its final position, so **nearly-sorted**
input runs close to O(n) even though it is not exactly O(n) input. This is
why insertion sort is the one simple sort that shows up inside real,
production sorting algorithms (e.g. as the base case for Timsort/introsort
on small subarrays).

**In-place:** yes. **Stable:** yes — the inner loop only shifts elements
strictly greater than the key, so an equal element is never moved past.


In [ ]:
def insertion_sort(arr):
    arr = arr[:]
    for i in range(1, len(arr)):
        key = arr[i]                          # the element being inserted
        j = i - 1
        while j >= 0 and arr[j] > key:        # strictly greater: stability preserved
            arr[j + 1] = arr[j]                # shift right to make room
            j -= 1
        arr[j + 1] = key                       # drop key into the gap left behind
    return arr


assert insertion_sort([5, 2, 4, 1, 3]) == [1, 2, 3, 4, 5]
assert insertion_sort([1, 2, 3, 5, 4]) == [1, 2, 3, 4, 5]   # nearly sorted: few shifts
assert insertion_sort([]) == []
assert insertion_sort([1, 2, 3, 4, 5]) == [1, 2, 3, 4, 5]


def count_inversions(arr):
    """Count out-of-order pairs (i < j but arr[i] > arr[j]) — a direct
    measure of 'how far from sorted' an array is, and (up to a constant
    factor) how many shifts insertion sort will perform."""
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if arr[i] > arr[j]:
                count += 1
    return count


assert count_inversions([1, 2, 3, 5, 4]) == 1          # one adjacent swap needed
assert count_inversions([6, 5, 4, 3, 2, 1]) == 15       # n(n-1)/2 for n=6: every pair inverted
assert count_inversions([1, 2, 3, 4, 5]) == 0
print("Insertion sort and inversion counting checks passed")


## 4. Empirically measuring comparisons and swaps

Rather than trust the O(n^2) formula on faith, let's instrument each sort to
literally count comparisons and swaps, then run them on the same random
input and print the real numbers. Expect selection sort's comparison count
to be close to n(n-1)/2 regardless of input order, and its swap count to be
close to n; expect insertion sort's comparison count to shrink dramatically
on nearly-sorted input.


In [ ]:
import random

def bubble_sort_counted(arr):
    arr = arr[:]
    n = len(arr)
    comparisons = swaps = 0
    for i in range(n - 1):
        swapped = False
        for j in range(n - 1 - i):
            comparisons += 1
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swaps += 1
                swapped = True
        if not swapped:
            break
    return arr, comparisons, swaps


def selection_sort_counted(arr):
    arr = arr[:]
    n = len(arr)
    comparisons = swaps = 0
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            comparisons += 1
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
        swaps += 1
    return arr, comparisons, swaps


def insertion_sort_counted(arr):
    arr = arr[:]
    comparisons = shifts = 0
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if arr[j] > key:
                arr[j + 1] = arr[j]
                shifts += 1
                j -= 1
            else:
                break
        arr[j + 1] = key
    return arr, comparisons, shifts


random.seed(42)
data = [random.randint(0, 999) for _ in range(200)]
nearly_sorted = sorted(data)[:-5] + [data[0], data[1], data[2], data[3], data[4]]

for label, sample in (("random n=200", data), ("nearly sorted n=200", nearly_sorted)):
    _, bc, bs = bubble_sort_counted(sample)
    _, sc, ss = selection_sort_counted(sample)
    _, ic, ishifts = insertion_sort_counted(sample)
    print(f"{label}:")
    print(f"  bubble    comparisons={bc:6d} swaps={bs:6d}")
    print(f"  selection comparisons={sc:6d} swaps={ss:6d}")
    print(f"  insertion comparisons={ic:6d} shifts={ishifts:6d}")

# Sanity checks on the real numbers we just measured (not invented ones)
n = len(data)
assert selection_sort_counted(data)[1] == n * (n - 1) // 2   # selection's comparisons are input-independent
_, ic_random, _ = insertion_sort_counted(data)
_, ic_nearly, _ = insertion_sort_counted(nearly_sorted)
assert ic_nearly < ic_random   # nearly-sorted input genuinely does fewer comparisons
print("\nEmpirical checks passed: selection sort comparisons are input-independent;")
print("insertion sort does fewer comparisons on nearly-sorted input, as predicted.")


## 5. Counting sort: sorting without comparing

All three sorts above are **comparison sorts** — they decide order purely by
comparing pairs of elements, which limits them to O(n log n) at best (see
Week 4 Lecture 4's decision-tree proof). Counting sort sidesteps this by
exploiting extra information: if you know every key is an integer in
`[0, k]`, you can count how many times each key occurs, turn those counts
into prefix sums (giving each key's final starting position), and place
elements directly — no comparisons at all.

**Complexity:** O(n + k) time, O(k) extra space. This beats O(n log n) when
k = O(n), but becomes worse than a comparison sort when k is much larger
than n (e.g. sorting 100 numbers drawn from `[0, 10**9]`).

**Stability requires care:** scanning the *input* backward while placing
into the output (using each key's count as a countdown) is what preserves
the original relative order of equal keys — scanning forward would reverse
it.


In [ ]:
def counting_sort(arr, k):
    """Sort a list of integers in [0, k] in O(n + k) time."""
    count = [0] * (k + 1)
    for x in arr:
        count[x] += 1
    for i in range(1, k + 1):
        count[i] += count[i - 1]              # prefix sums: count[x] is now the number
                                               # of elements <= x, i.e. x's final index + 1
    output = [0] * len(arr)
    for x in reversed(arr):                   # backward scan: preserves stability
        count[x] -= 1
        output[count[x]] = x
    return output


assert counting_sort([2, 5, 3, 0, 2, 3], 5) == [0, 2, 2, 3, 3, 5]
assert counting_sort([], 5) == []


def counting_sort_tagged(pairs, k):
    """Same algorithm, but sorting (key, tag) pairs by key only, to make
    stability directly observable."""
    count = [0] * (k + 1)
    for key, _ in pairs:
        count[key] += 1
    for i in range(1, k + 1):
        count[i] += count[i - 1]
    output = [None] * len(pairs)
    for key, tag in reversed(pairs):
        count[key] -= 1
        output[count[key]] = (key, tag)
    return output


tagged = [(3, "a"), (3, "b"), (1, "c")]
result = counting_sort_tagged(tagged, 3)
assert result == [(1, "c"), (3, "a"), (3, "b")]   # "a" still precedes "b": stable
print("Counting sort checks passed, stability confirmed")


## 6. Radix sort: counting sort, digit by digit

Radix sort handles a much larger key range than plain counting sort would
allow, by sorting on one digit position at a time (least significant digit
first), using counting sort as the subroutine for each digit — which only
ever needs `k = 9` (base 10) buckets. Because counting sort is stable, and
because we process digits from least to most significant, each pass
preserves the correct relative order established by the previous, less
significant passes.

**Complexity:** O(d * (n + b)) where `d` is the number of digits and `b` is
the base (10 here). For fixed-width integers, `d` is a constant, so this is
effectively O(n).


In [ ]:
def counting_sort_by_digit(arr, exp):
    """Stably sort `arr` by the digit at place value `exp` (1, 10, 100, ...)."""
    n = len(arr)
    output = [0] * n
    count = [0] * 10                          # base 10: digits 0-9
    for x in arr:
        digit = (x // exp) % 10
        count[digit] += 1
    for i in range(1, 10):
        count[i] += count[i - 1]
    for x in reversed(arr):                   # backward again: stability across passes
        digit = (x // exp) % 10
        count[digit] -= 1
        output[count[digit]] = x
    return output


def radix_sort(arr):
    if not arr:
        return arr
    max_val = max(arr)
    exp = 1
    result = arr[:]
    while max_val // exp > 0:                 # one pass per digit position, LSD first
        result = counting_sort_by_digit(result, exp)
        exp *= 10
    return result


assert radix_sort([170, 45, 75, 90, 802, 24, 2]) == [2, 24, 45, 75, 90, 170, 802]
assert radix_sort([]) == []
assert radix_sort([5]) == [5]

import random
random.seed(7)
random_data = [random.randint(0, 999999) for _ in range(2000)]
assert radix_sort(random_data) == sorted(random_data)
print("Radix sort checks passed, matching Python's built-in sorted()")


## 7. Comparing all four sorts, and when "simple" wins

| Sort | Best | Average | Worst | Stable | In-place | Notes |
|---|---|---|---|---|---|---|
| Bubble | O(n) | O(n^2) | O(n^2) | Yes | Yes | Early-exit flag helps only on nearly-sorted input |
| Selection | O(n^2) | O(n^2) | O(n^2) | No | Yes | Comparisons are always ~n^2/2, but only O(n) swaps |
| Insertion | O(n) | O(n^2) | O(n^2) | Yes | Yes | Excellent on small or nearly-sorted input |
| Counting | O(n+k) | O(n+k) | O(n+k) | Yes | No | Needs integer keys in a known, small range `[0,k]` |
| Radix | O(d(n+b)) | O(d(n+b)) | O(d(n+b)) | Yes | No | Extends counting sort to a larger key range |

Despite all being O(n^2) in the worst case, the O(n log n) algorithms in
Week 5 do not automatically win everywhere:

- **Very small n** (roughly n < 20-30): the constant-factor overhead of
  recursive divide-and-conquer sorts can make insertion sort faster in
  absolute time, which is why production sorts (Timsort, introsort) fall
  back to insertion sort for small subarrays.
- **Nearly-sorted data:** insertion sort's cost tracks the number of
  inversions, so it can beat an O(n log n) sort that does not exploit
  existing order.
- **Small, bounded integer keys** (e.g. exam scores 0-100, single-digit
  grades): counting sort's O(n+k) beats any comparison sort outright.
- **Memory-constrained settings:** the simple sorts are all in-place,
  unlike merge sort's O(n) auxiliary array (see Week 5).


In [ ]:
import timeit

# Empirically confirm insertion sort's edge on nearly-sorted data, and its
# disadvantage on random data, against Python's own highly-optimized sorted().
random.seed(1)
random_1000 = [random.randint(0, 9999) for _ in range(1000)]
nearly_sorted_1000 = sorted(random_1000)
# perturb just 10 elements out of position to keep it "nearly" sorted
for _ in range(10):
    i, j = random.randint(0, 999), random.randint(0, 999)
    nearly_sorted_1000[i], nearly_sorted_1000[j] = nearly_sorted_1000[j], nearly_sorted_1000[i]

t_insertion_random = timeit.timeit(lambda: insertion_sort(random_1000), number=3)
t_insertion_nearly = timeit.timeit(lambda: insertion_sort(nearly_sorted_1000), number=3)

print(f"insertion_sort on random     n=1000: {t_insertion_random:.4f}s")
print(f"insertion_sort on nearly-sorted n=1000: {t_insertion_nearly:.4f}s")
assert t_insertion_nearly < t_insertion_random   # nearly-sorted must be faster, measured live
print("Confirmed: insertion sort is measurably faster on nearly-sorted input")


## 8. Linear search

Linear search is the simplest possible search: scan the array from the
front, comparing each element to the target, and stop as soon as a match is
found (or the array runs out). It requires **no precondition** on the
input's order — it works on an unsorted array just as well as a sorted one,
which is exactly why it is still used in practice even though faster
alternatives exist for sorted data.

**Complexity:** O(1) best case (the target is the very first element), O(n)
worst case (the target is last, or absent entirely — every element must be
checked), and O(n) average case too (on average, about n/2 comparisons for a
present target, and always n comparisons to confirm absence). **Extra
space:** O(1). **Precondition:** none.


In [ ]:
def linear_search(arr, target):
    """Return the index of the first occurrence of target in arr, or -1 if
    target is not present. Works on unsorted input.

    Args:
        arr: list of comparable elements, in any order.
        target: the value to search for.
    Returns:
        The index of the first occurrence of target, or -1 if not found.
    """
    for i, x in enumerate(arr):        # no ordering assumed: must check every element
        if x == target:
            return i                   # stop as soon as a match is found
    return -1                          # ran off the end: target is not present


assert linear_search([5, 2, 4, 1, 3], 4) == 2
assert linear_search([5, 2, 4, 1, 3], 9) == -1
assert linear_search([], 1) == -1
assert linear_search([7, 7, 7], 7) == 0        # first occurrence, not any occurrence
print("Linear search checks passed")


## 9. Binary search

Binary search exploits a **precondition that linear search does not need:
the array must already be sorted**. Each step compares the target to the
middle element and discards the half of the array that cannot possibly
contain it, so the search space shrinks by half every comparison. This
divide-and-conquer shrinkage is what gives binary search its O(log n) bound
— doubling the input size costs only one extra comparison.

**Complexity:** O(log n) worst and average case, O(1) best case (target is
exactly the first middle element checked). **Precondition:** the array must
be sorted ascending. **Extra space:** O(1) iterative, O(log n) recursive (see
the stack-depth discussion below).

### Classic pitfalls

- **Off-by-one in the loop condition.** The loop must use `low <= high`, not
  `low < high`. With `<`, the case `low == high` (exactly one element left to
  check) is skipped, so a target that happens to be the very last remaining
  candidate is never actually compared — the search incorrectly reports
  "not found."
- **Off-by-one in the bound updates.** After checking `mid` and ruling it
  out, the new bound must exclude `mid` itself: `low = mid + 1` or
  `high = mid - 1`, never `low = mid` or `high = mid`. Forgetting the `+ 1`
  / `- 1` can leave `mid` permanently re-checked, causing an infinite loop
  because `low` and `high` never converge.
- **Integer overflow when computing `mid`.** The intuitive formula
  `mid = (low + high) // 2` computes `low + high` before dividing. In a
  language with fixed-width integers (C, Java, C++), if `low` and `high` are
  both large, `low + high` can silently overflow the integer type and wrap
  around to a negative number, corrupting the search (this was a real,
  famous bug in binary search implementations for years — see Joshua
  Bloch's 2006 "Extra, Extra" writeup on the Java binary search bug). The
  fix is `mid = low + (high - low) // 2`, which never lets the intermediate
  sum exceed `high`. **Python integers are arbitrary-precision, so this
  specific overflow cannot happen here** — but the safer formula is worth
  knowing and using out of habit, since most other languages you will use
  professionally (C, C++, Java, Go) do not have this luxury.

### Recursive vs iterative: the space-complexity tradeoff

The iterative version uses a `while` loop and two integer variables, so its
auxiliary space is O(1) regardless of array size. The recursive version
makes one recursive call per halving of the search space, and each call
adds a frame to the call stack that is not popped until that call returns
— so the maximum stack depth is O(log n), meaning O(log n) auxiliary space,
not O(1). For a typical search this difference is negligible (log2 of even
a billion elements is only about 30), but it matters in recursion-limited
environments (Python's default recursion limit, or a fixed-size call stack
in embedded/systems code) and is a standard interview follow-up question:
"can you do this without recursion?"


In [ ]:
def binary_search_iterative(arr, target):
    """Return the index of target in the sorted list arr, or -1 if absent.
    Iterative, O(log n) time, O(1) auxiliary space.

    Args:
        arr: list of comparable elements, sorted ascending.
        target: the value to search for.
    Returns:
        The index of target if present, else -1.
    """
    low, high = 0, len(arr) - 1
    while low <= high:                          # "<=" is required: see off-by-one discussion above
        mid = low + (high - low) // 2           # overflow-safe form; matters in C/Java, not Python
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1                       # discard the left half, including mid itself
        else:
            high = mid - 1                      # discard the right half, including mid itself
    return -1


def binary_search_recursive(arr, target, low=0, high=None):
    """Return the index of target in the sorted list arr, or -1 if absent.
    Recursive, O(log n) time, O(log n) auxiliary space (call stack depth).

    Args:
        arr: list of comparable elements, sorted ascending.
        target: the value to search for.
        low, high: current search bounds (inclusive); default to the full array.
    Returns:
        The index of target if present, else -1.
    """
    if high is None:
        high = len(arr) - 1
    if low > high:                              # base case: search space is empty
        return -1
    mid = low + (high - low) // 2
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, high)
    else:
        return binary_search_recursive(arr, target, low, mid - 1)


sorted_data = [1, 3, 4, 6, 8, 9, 11, 15, 20, 25]
for finder in (binary_search_iterative, binary_search_recursive):
    assert finder(sorted_data, 1) == 0            # first element (best case for iterative)
    assert finder(sorted_data, 25) == 9            # last element
    assert finder(sorted_data, 8) == 4             # middle-ish element
    assert finder(sorted_data, 7) == -1            # absent, between two present values
    assert finder([], 5) == -1                     # empty array
    assert finder([5], 5) == 0                     # single element, present
    assert finder([5], 3) == -1                    # single element, absent

print("Binary search checks passed (iterative and recursive agree)")

# Confirm iterative and recursive give identical answers across many probes,
# and cross-check against linear search on the same sorted data.
import random
random.seed(11)
probe_data = sorted(random.sample(range(0, 2000), 300))
for _ in range(200):
    probe = random.choice(probe_data + [random.randint(-10, 2010)])
    i_result = binary_search_iterative(probe_data, probe)
    r_result = binary_search_recursive(probe_data, probe)
    l_result = linear_search(probe_data, probe)
    assert i_result == r_result                    # iterative and recursive always agree
    # both report "found" or "not found" consistently with linear search
    assert (i_result != -1) == (l_result != -1)
print("Cross-check against linear search passed on 200 random probes")


## Exercises

Try each one yourself before checking the Solutions section at the end.


**Exercise 1 — `cocktail_shaker_sort`.** Implement cocktail shaker sort
(bidirectional bubble sort): like bubble sort, but alternating direction on
each pass — one pass left-to-right bubbling the maximum to the end, the next
pass right-to-left bubbling the minimum to the front. Stop early if a pass
(in either direction) makes no swaps.

Example: `[5, 1, 4, 2, 8, 0, 2]` -> `[0, 1, 2, 2, 4, 5, 8]`.


In [ ]:
def cocktail_shaker_sort(arr):
    """Sort arr ascending using cocktail shaker (bidirectional bubble) sort.
    Returns a new sorted list; does not mutate the input.

    Args:
        arr: list of comparable elements.
    Returns:
        A new list containing the same elements in ascending order.
    """
    # TODO: implement this. Hint: keep `start` and `end` bounds that shrink
    # from both sides, alternate a forward pass (bubbling the max to `end`)
    # and a backward pass (bubbling the min to `start`), and track a
    # `swapped` flag across BOTH passes to allow early exit.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 1
assert cocktail_shaker_sort([5, 1, 4, 2, 8, 0, 2]) == [0, 1, 2, 2, 4, 5, 8]
assert cocktail_shaker_sort([]) == []
assert cocktail_shaker_sort([1]) == [1]
assert cocktail_shaker_sort([3, 2, 1]) == [1, 2, 3]
assert cocktail_shaker_sort([1, 2, 3]) == [1, 2, 3]
print("Exercise 1 passed")


**Exercise 2 — `is_sorted_and_stable_check`.** Given an *original* list of
`(key, tag)` pairs and a *sorted* list of the same pairs (sorted by `key`),
return `True` if the sorted list is both correctly sorted by key AND stable
with respect to the original (i.e. for every pair of equal keys, their
relative tag order from `original` is preserved in `sorted_pairs`), else
`False`. This is the kind of check you'd use to verify any sort's stability
automatically, instead of eyeballing it.

Example: `original=[(3,'a'),(3,'b'),(1,'c')]`,
`sorted_pairs=[(1,'c'),(3,'a'),(3,'b')]` -> `True`.
`sorted_pairs=[(1,'c'),(3,'b'),(3,'a')]` -> `False` (unstable order).


In [ ]:
def is_sorted_and_stable_check(original, sorted_pairs):
    """Check that sorted_pairs is a correctly sorted (by key, ascending)
    and stable rearrangement of original.

    Args:
        original: list of (key, tag) tuples, in original order.
        sorted_pairs: list of the same (key, tag) tuples, claimed sorted.
    Returns:
        True if sorted_pairs is sorted ascending by key AND, for every
        group of equal keys, the tags appear in the same relative order
        as in `original`. False otherwise.
    """
    # TODO: implement this. Hint: (1) check keys are non-decreasing across
    # sorted_pairs; (2) for each distinct key, extract the tags for that key
    # from `original` in order, and from `sorted_pairs` in order, and check
    # the two tag-lists are equal.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 2
original = [(3, "a"), (3, "b"), (1, "c")]
stable_sorted = [(1, "c"), (3, "a"), (3, "b")]
unstable_sorted = [(1, "c"), (3, "b"), (3, "a")]
wrongly_ordered = [(3, "a"), (1, "c"), (3, "b")]

assert is_sorted_and_stable_check(original, stable_sorted) is True
assert is_sorted_and_stable_check(original, unstable_sorted) is False
assert is_sorted_and_stable_check(original, wrongly_ordered) is False
assert is_sorted_and_stable_check([], []) is True
print("Exercise 2 passed")


**Exercise 3 — `insertion_sort_with_binary_search`.** Standard insertion
sort finds the insertion point for `key` by scanning linearly, which costs
O(n) comparisons per insertion even though the prefix is already sorted (and
binary search could find the spot in O(log n) comparisons). Implement
*binary insertion sort*: use `bisect.bisect_right` (or your own binary
search) to find the insertion index in the sorted prefix, then shift
elements to make room. Note this only reduces *comparisons*, not *shifts* —
shifting still costs O(n) in the worst case, so overall time complexity
stays O(n^2), but the comparison count drops to O(n log n).

Example: `[5, 2, 4, 1, 3]` -> `[1, 2, 3, 4, 5]`.


In [ ]:
import bisect

def insertion_sort_with_binary_search(arr):
    """Sort arr ascending using insertion sort, but locate each element's
    insertion point in the sorted prefix via binary search instead of a
    linear scan. Returns a new list; does not mutate the input.

    Args:
        arr: list of comparable elements.
    Returns:
        A new list containing the same elements in ascending order.
    """
    # TODO: implement this. Hint: maintain arr[:i] as the sorted prefix
    # (copy the input first). For each new key at position i, use
    # bisect.bisect_right(sorted_copy, key, 0, i) to find where it belongs,
    # then use list.insert or manual shifting to place it there. Remember
    # bisect_right (not bisect_left) is what keeps the sort stable.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 3
assert insertion_sort_with_binary_search([5, 2, 4, 1, 3]) == [1, 2, 3, 4, 5]
assert insertion_sort_with_binary_search([]) == []
assert insertion_sort_with_binary_search([1]) == [1]
assert insertion_sort_with_binary_search([3, 3, 1, 1, 2]) == [1, 1, 2, 3, 3]

random.seed(3)
big = [random.randint(0, 500) for _ in range(150)]
assert insertion_sort_with_binary_search(big) == sorted(big)
print("Exercise 3 passed")


**Exercise 4 (harder) — `counting_sort_with_offset`.** Plain counting
sort as written in Section 5 only handles non-negative keys starting at 0.
Extend it to handle any list of integers, including negative ones, by
computing the actual `min` and `max` of the input and offsetting keys into a
`[0, max-min]` range internally, returning results in the original key
values (not the shifted ones).

Example: `[-5, 3, -1, 0, 3, -5]` -> `[-5, -5, -1, 0, 3, 3]`.


In [ ]:
def counting_sort_with_offset(arr):
    """Sort a list of (possibly negative) integers using counting sort,
    by offsetting keys into a non-negative range internally.

    Args:
        arr: list of ints (may be empty, may include negative numbers).
    Returns:
        A new list with the same ints in ascending order.
    """
    # TODO: implement this. Hint: if arr is empty return []. Otherwise
    # compute lo = min(arr), hi = max(arr), k = hi - lo, build a count
    # array of size k+1 indexed by (x - lo), do the same prefix-sum and
    # backward-placement steps as counting_sort, then add lo back to each
    # placed value (or just place x directly using the shifted index).
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 4
assert counting_sort_with_offset([-5, 3, -1, 0, 3, -5]) == [-5, -5, -1, 0, 3, 3]
assert counting_sort_with_offset([]) == []
assert counting_sort_with_offset([7]) == [7]
assert counting_sort_with_offset([2, 2, 2]) == [2, 2, 2]

random.seed(9)
neg_data = [random.randint(-1000, 1000) for _ in range(300)]
assert counting_sort_with_offset(neg_data) == sorted(neg_data)
print("Exercise 4 passed")


**Exercise 5 — `count_occurrences_linear`.** Given a list (not
necessarily sorted) and a target value, use a linear scan to count *how
many times* `target` appears in `arr` — not just whether it is present.
This is the kind of query where linear search is the *only* option once the
array is unsorted, since binary search requires sorted order and only finds
a single boundary, not a count, without extra machinery.

Example: `count_occurrences_linear([4, 2, 4, 4, 1], 4)` -> `3`.


In [ ]:
def count_occurrences_linear(arr, target):
    """Count how many times target appears in arr, using a linear scan.

    Args:
        arr: list of comparable elements, in any order.
        target: the value to count.
    Returns:
        The number of times target appears in arr (0 if absent).
    """
    # TODO: implement this. Hint: walk the list once, incrementing a counter
    # every time an element equals target. No sorting or early exit needed
    # (an unsorted array offers no way to know occurrences have run out).
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 5
assert count_occurrences_linear([4, 2, 4, 4, 1], 4) == 3
assert count_occurrences_linear([1, 2, 3], 9) == 0
assert count_occurrences_linear([], 1) == 0
assert count_occurrences_linear([5, 5, 5, 5], 5) == 4
print("Exercise 5 passed")


**Exercise 6 — `find_first_occurrence_sorted`.** A sorted array can
contain duplicate values, and plain binary search only guarantees finding
*some* index where `target` occurs, not necessarily the first one.
Implement a variant that returns the index of the **first** (leftmost)
occurrence of `target`, or `-1` if absent, in O(log n) time — by continuing
to search the left half even after finding a match, instead of returning
immediately.

Example: `find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 2)` -> `1`.


In [ ]:
def find_first_occurrence_sorted(arr, target):
    """Return the index of the FIRST (leftmost) occurrence of target in the
    sorted list arr, or -1 if target is not present. Must run in O(log n).

    Args:
        arr: list of comparable elements, sorted ascending, may contain
            duplicates.
        target: the value to search for.
    Returns:
        The smallest index i such that arr[i] == target, or -1 if none.
    """
    # TODO: implement this. Hint: adapt binary_search_iterative, but when
    # arr[mid] == target, do NOT return immediately -- instead record mid as
    # a candidate answer and keep searching the LEFT half (high = mid - 1)
    # in case an earlier occurrence exists. Return the best candidate found,
    # or -1 if none was ever recorded.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 6
assert find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 2) == 1
assert find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 8) == 5
assert find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 9) == -1
assert find_first_occurrence_sorted([], 1) == -1
assert find_first_occurrence_sorted([2, 2, 2, 2], 2) == 0
print("Exercise 6 passed")


## Quiz

**Q1.** Selection sort always performs about n(n-1)/2 comparisons, regardless
of whether the input is already sorted, reverse sorted, or random. Why
doesn't selection sort have a fast best case the way bubble sort and
insertion sort do?

<details><summary>Show answer</summary>
Selection sort's inner loop always scans the *entire* unsorted remainder to
find its minimum, because it has no way to know the remainder is already in
order without comparing every element — there is no analogue of bubble
sort's "no swaps this pass" signal or insertion sort's "key already belongs
here" early stop. The cost is baked into finding the minimum, not into
fixing misplaced elements, so input order does not help.
</details>

**Q2.** Counting sort and radix sort are not comparison sorts, yet the
Week 4 Lecture 4 proof shows any comparison sort needs at least
`ceil(log2(n!))` comparisons — roughly O(n log n). Why doesn't sorting
100 numbers from `[0, 999]` with counting sort in O(n + k) time violate that
lower bound?

<details><summary>Show answer</summary>
The lower bound applies only to algorithms that determine order solely by
pairwise comparisons ("comparison sorts"). Counting sort never compares two
elements to each other at all — it uses the numeric *value* of each key
directly as an array index. Since it exploits extra structure (keys are
integers in a known small range) that a general comparison sort is not
allowed to assume, the proof simply does not apply to it.
</details>

**Q3.** Is selection sort stable? Justify your answer using the tagged
example from Section 2.

<details><summary>Show answer</summary>
No. In the tagged example `[(3,"a"), (3,"b"), (1,"c")]`, selection sort
finds the minimum `(1,"c")` at index 2 and swaps it directly with index 0's
`(3,"a")`. That swap moves `(3,"a")` to index 2, jumping it past `(3,"b")`
at index 1 — so the two equal-keyed elements end up in the order
`(3,"b"), (3,"a")`, reversed from their original relative order. That is
exactly what "not stable" means.
</details>

**Q4.** For sorting an array of 10 million exam scores, each an integer from
0 to 100, which of the sorts covered in this notebook is the best choice,
and why?

<details><summary>Show answer</summary>
Counting sort. The key range k=100 is tiny and fixed regardless of n, so
counting sort runs in O(n + k) = O(n) time — strictly better than any
comparison sort's O(n log n) floor, and dramatically better than any of the
simple O(n^2) sorts. This is precisely Week 4 Lecture 3's "exam grade tally"
scenario.
</details>

**Q5.** You are given a sorted array of 1 billion elements and need to check
whether a single specific value is present. Would you use linear search or
binary search, and how many comparisons (roughly) does each need in the
worst case?

<details><summary>Show answer</summary>
Binary search. Linear search needs up to 1,000,000,000 comparisons in the
worst case (O(n)). Binary search needs only about log2(1,000,000,000) ≈ 30
comparisons in the worst case (O(log n)) — a difference of eight orders of
magnitude. The only reason to prefer linear search here would be if the
array were not sorted and sorting it first would cost more than the linear
scans you expect to do.
</details>

**Q6.** A student writes `mid = (low + high) // 2` inside a binary search
in Python and argues this is fine because "integers can't overflow in
Python." Are they right, and would the same code be safe if translated
directly into Java?

<details><summary>Show answer</summary>
They are right about Python: Python's `int` is arbitrary-precision, so
`low + high` simply grows as large as needed and `(low + high) // 2` is
always correct, no matter how large the array. They would NOT be right in
Java (or C/C++): `int` there is a fixed-width (typically 32-bit) type, so if
`low` and `high` are both large enough that their sum exceeds the maximum
representable `int`, `low + high` silently overflows and wraps to a
negative number, breaking the search in a way that is very hard to spot
because it only manifests on large arrays. The overflow-safe form
`mid = low + (high - low) // 2` avoids this because the intermediate value
`high - low` is always at most `high`, so the sum never approaches the
overflow boundary. This is why the safer formula is worth using as a habit
even in Python, where it happens not to matter.
</details>


## Solutions (try the exercises yourself first!)


In [ ]:
# Solution: Exercise 1
def cocktail_shaker_sort(arr):
    arr = arr[:]
    start, end = 0, len(arr) - 1
    swapped = True
    while swapped:
        swapped = False
        for i in range(start, end):               # forward pass: bubble max to `end`
            if arr[i] > arr[i + 1]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                swapped = True
        if not swapped:
            break
        end -= 1
        swapped = False
        for i in range(end - 1, start - 1, -1):    # backward pass: bubble min to `start`
            if arr[i] > arr[i + 1]:
                arr[i], arr[i + 1] = arr[i + 1], arr[i]
                swapped = True
        start += 1
    return arr

assert cocktail_shaker_sort([5, 1, 4, 2, 8, 0, 2]) == [0, 1, 2, 2, 4, 5, 8]
assert cocktail_shaker_sort([]) == []
print("Exercise 1 solution verified")


In [ ]:
# Solution: Exercise 2
def is_sorted_and_stable_check(original, sorted_pairs):
    keys = [k for k, _ in sorted_pairs]
    if keys != sorted(keys):
        return False
    from collections import defaultdict
    original_tags = defaultdict(list)
    for k, tag in original:
        original_tags[k].append(tag)
    sorted_tags = defaultdict(list)
    for k, tag in sorted_pairs:
        sorted_tags[k].append(tag)
    return original_tags == sorted_tags

original = [(3, "a"), (3, "b"), (1, "c")]
assert is_sorted_and_stable_check(original, [(1, "c"), (3, "a"), (3, "b")]) is True
assert is_sorted_and_stable_check(original, [(1, "c"), (3, "b"), (3, "a")]) is False
print("Exercise 2 solution verified")


In [ ]:
# Solution: Exercise 3
import bisect

def insertion_sort_with_binary_search(arr):
    result = arr[:]
    for i in range(1, len(result)):
        key = result[i]
        pos = bisect.bisect_right(result, key, 0, i)   # bisect_right preserves stability
        result.pop(i)
        result.insert(pos, key)
    return result

assert insertion_sort_with_binary_search([5, 2, 4, 1, 3]) == [1, 2, 3, 4, 5]
assert insertion_sort_with_binary_search([3, 3, 1, 1, 2]) == [1, 1, 2, 3, 3]
print("Exercise 3 solution verified")


In [ ]:
# Solution: Exercise 4
def counting_sort_with_offset(arr):
    if not arr:
        return []
    lo, hi = min(arr), max(arr)
    k = hi - lo
    count = [0] * (k + 1)
    for x in arr:
        count[x - lo] += 1
    for i in range(1, k + 1):
        count[i] += count[i - 1]
    output = [0] * len(arr)
    for x in reversed(arr):
        idx = x - lo
        count[idx] -= 1
        output[count[idx]] = x
    return output

assert counting_sort_with_offset([-5, 3, -1, 0, 3, -5]) == [-5, -5, -1, 0, 3, 3]
assert counting_sort_with_offset([]) == []
print("Exercise 4 solution verified")


In [ ]:
# Solution: Exercise 5
def count_occurrences_linear(arr, target):
    count = 0
    for x in arr:
        if x == target:
            count += 1
    return count

assert count_occurrences_linear([4, 2, 4, 4, 1], 4) == 3
assert count_occurrences_linear([1, 2, 3], 9) == 0
print("Exercise 5 solution verified")


In [ ]:
# Solution: Exercise 6
def find_first_occurrence_sorted(arr, target):
    low, high = 0, len(arr) - 1
    result = -1
    while low <= high:
        mid = low + (high - low) // 2
        if arr[mid] == target:
            result = mid           # record this match...
            high = mid - 1         # ...but keep looking to its left for an earlier one
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return result

assert find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 2) == 1
assert find_first_occurrence_sorted([1, 2, 2, 2, 5, 8], 9) == -1
print("Exercise 6 solution verified")


## MTech Extension — Bucket Sort and a Formal Average-Case Analysis

**Bucket sort** generalizes counting sort's idea to non-integer, real-valued
keys drawn (roughly) uniformly from a known range: partition `[min, max)`
into `m` equal-width buckets, scatter each element into its bucket by value,
sort each bucket with a simple sort like insertion sort (buckets are
expected to be small), then concatenate the buckets in order.

**Average-case analysis (formal):** if `n` keys are drawn independently and
uniformly from `[0, 1)` and distributed into `n` buckets of width `1/n`,
the expected number of elements landing in any given bucket is 1 (a
balls-into-bins argument), and more precisely the bucket sizes follow a
distribution with expected sum-of-squares `E[sum(n_i^2)] = 2n - 1` for `n`
balls into `n` bins. Since insertion-sorting each bucket costs O(n_i^2) in
the worst case per bucket, the *expected* total sorting cost across all
buckets is `O(sum(n_i^2)) = O(n)`, and scattering plus concatenation is also
O(n) — giving **expected O(n)** total time, strictly better than any
comparison sort's O(n log n) floor, precisely because bucket sort (like
counting and radix sort) is not a comparison sort. Unlike counting sort's
worst case, bucket sort's guarantee genuinely depends on the *uniformity*
assumption: adversarial or heavily skewed input (e.g. all keys clustered in
one bucket) degrades it toward O(n^2), since one bucket then holds
essentially all n elements.


In [ ]:
def insertion_sort(values):
    values = values[:]
    for i in range(1, len(values)):
        key = values[i]
        j = i - 1
        while j >= 0 and values[j] > key:
            values[j + 1] = values[j]
            j -= 1
        values[j + 1] = key
    return values


def bucket_sort(arr, num_buckets=None):
    """Sort a list of floats in [0, 1) using bucket sort. Expected O(n)
    time when values are roughly uniformly distributed."""
    if not arr:
        return []
    n = len(arr)
    if num_buckets is None:
        num_buckets = n                    # one bucket per element: expected load 1 each
    buckets = [[] for _ in range(num_buckets)]
    for x in arr:
        idx = min(int(x * num_buckets), num_buckets - 1)   # clamp x==1.0's edge case
        buckets[idx].append(x)
    result = []
    for bucket in buckets:
        result.extend(insertion_sort(bucket))   # each bucket is small: insertion sort is cheap here
    return result


import random
random.seed(2)
uniform_data = [random.random() for _ in range(5000)]
sorted_result = bucket_sort(uniform_data)
assert sorted_result == sorted(uniform_data)

# Empirically confirm the balls-into-bins expectation: with n buckets for n
# uniform items, average bucket occupancy should be close to 1.
n = 5000
buckets_count = [0] * n
for x in uniform_data:
    idx = min(int(x * n), n - 1)
    buckets_count[idx] += 1
average_occupancy = sum(buckets_count) / n
assert 0.9 < average_occupancy < 1.1   # measured, not assumed: should hover near 1.0
print(f"Average bucket occupancy over {n} buckets / {n} items: {average_occupancy:.3f} (expected ~1.0)")

# Contrast: skewed (non-uniform) input degrades bucket sort's advantage,
# because one bucket ends up holding almost everything.
skewed_data = [random.uniform(0, 0.01) for _ in range(2000)]   # all crammed into the first 1% of the range
skewed_buckets = [0] * 2000
for x in skewed_data:
    idx = min(int(x * 2000), 1999)
    skewed_buckets[idx] += 1
max_occupancy = max(skewed_buckets)
assert max_occupancy > 100   # far above the uniform expectation of ~1: the assumption has broken down
print(f"Skewed input: max single-bucket occupancy = {max_occupancy} (uniform assumption violated)")
